####**customers**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import DataFrame
from pyspark.sql import window

In [0]:
import os
import sys

In [0]:

current_dir= os.getcwd()
sys.path.append(current_dir)

In [0]:
df=spark.read.table('pysparkdbt.bronze.customers')

In [0]:
df_cust = df.withColumn("domain", split(col('email'),'@')[1])


# display(df_cust)

In [0]:
df_cust = df_cust.withColumn("phone_number",regexp_replace("phone_number",r"[^0-9]",""))
# display(df_cust)

In [0]:
df_cust= df_cust.withColumn("full_name",concat_ws(" ",col('first_name'),col('last_name')))

df_cust = df_cust.drop('first_name','last_name')
# display(df_cust)

In [0]:
spark.sql("DROP TABLE IF EXISTS pysparkdbt.silver.customers")

DataFrame[]

In [0]:
df_cust.printSchema()
# first_name and last_name should NOT appear

root
 |-- customer_id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- last_updated_timestamp: timestamp (nullable = true)
 |-- domain: string (nullable = true)
 |-- full_name: string (nullable = false)



In [0]:
import importlib
import utils.custom_utils as cu
importlib.reload(cu)
from utils.custom_utils import transformations

In [0]:
cust_obj = transformations(spark)

custom_df_transform = cust_obj.dedup(df_cust,['customer_id'],'last_updated_timestamp')

# display(custom_df_transform)

In [0]:
df_cust=cust_obj.process_timestamp(custom_df_transform)
# display(df_cust)

In [0]:

#spark.sql("DROP TABLE IF EXISTS pysparkdbt.silver.customers")
spark.sql("DESCRIBE pysparkdbt.silver.customers").show()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|         customer_id|      int|   NULL|
|               email|   string|   NULL|
|        phone_number|   string|   NULL|
|                city|   string|   NULL|
|         signup_date|     date|   NULL|
|last_updated_time...|timestamp|   NULL|
|              domain|   string|   NULL|
|           full_name|   string|   NULL|
|   process_timestamp|timestamp|   NULL|
+--------------------+---------+-------+



In [0]:
from delta.tables import DeltaTable
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()  # ADD THIS


if not spark.catalog.tableExists('pysparkdbt.silver.customers'):
    # ✅ df_cust not df
    df_cust.write.format('delta')\
        .mode('overwrite')\
        .option("overwriteSchema", "true")\
        .saveAsTable("pysparkdbt.silver.customers")
    print("Table created successfully")
else:
    cust_obj.upsert(df_cust, ["customer_id"], 'customers', 'last_updated_timestamp')
    print("Upsert done")

Table created successfully


###***Drivers***


In [0]:
df_drivers=spark.read.table('pysparkdbt.bronze.drivers')

In [0]:
display(df_drivers)

driver_id,first_name,last_name,phone_number,vehicle_id,driver_rating,city,last_updated_timestamp
1,Latasha,Lopez,262-924-2955x590,1,4.7,East Dorothy,2025-08-25T06:36:26.000Z
2,Alan,Wiley,0967969634,2,3.98,West Susan,2025-09-14T00:44:57.000Z
3,James,Taylor,424-614-1847,3,3.66,Mcintoshton,2025-08-26T22:28:17.000Z
4,Theresa,Benson,617-017-0101x91777,4,3.86,North Courtneychester,2025-09-01T11:40:55.000Z
5,Karen,Jensen,611-060-5683,5,4.87,Brownburgh,2025-09-04T16:35:04.000Z
6,Debra,Smith,556.480.9096x439,6,4.26,Port Williamland,2025-08-31T14:31:37.000Z
7,Justin,Peters,+1-798-568-6952x9778,7,3.67,West Erinborough,2025-09-17T05:57:45.000Z
8,Todd,Young,706.321.8390x08097,8,4.9,Lake Stephen,2025-08-26T06:45:51.000Z
9,Mary,Young,(172)791-0504x5499,9,4.5,West Lindsey,2025-08-27T13:04:32.000Z
10,Jacob,Mack,(509)613-4480,10,4.04,Lauraland,2025-09-09T05:50:23.000Z


In [0]:
df_drivers = df_drivers.withColumn("phone_number",regexp_replace("phone_number",r"[^0-9]",""))
display(df_drivers)

driver_id,first_name,last_name,phone_number,vehicle_id,driver_rating,city,last_updated_timestamp
1,Latasha,Lopez,2629242955590,1,4.7,East Dorothy,2025-08-25T06:36:26.000Z
2,Alan,Wiley,0967969634,2,3.98,West Susan,2025-09-14T00:44:57.000Z
3,James,Taylor,4246141847,3,3.66,Mcintoshton,2025-08-26T22:28:17.000Z
4,Theresa,Benson,617017010191777,4,3.86,North Courtneychester,2025-09-01T11:40:55.000Z
5,Karen,Jensen,6110605683,5,4.87,Brownburgh,2025-09-04T16:35:04.000Z
6,Debra,Smith,5564809096439,6,4.26,Port Williamland,2025-08-31T14:31:37.000Z
7,Justin,Peters,179856869529778,7,3.67,West Erinborough,2025-09-17T05:57:45.000Z
8,Todd,Young,706321839008097,8,4.9,Lake Stephen,2025-08-26T06:45:51.000Z
9,Mary,Young,17279105045499,9,4.5,West Lindsey,2025-08-27T13:04:32.000Z
10,Jacob,Mack,5096134480,10,4.04,Lauraland,2025-09-09T05:50:23.000Z


In [0]:
df_drivers= df_drivers.withColumn("full_name",concat_ws(" ",col('first_name'),col('last_name')))

df_drivers = df_drivers.drop('first_name','last_name')
display(df_drivers)

driver_id,phone_number,vehicle_id,driver_rating,city,last_updated_timestamp,full_name
1,2629242955590,1,4.7,East Dorothy,2025-08-25T06:36:26.000Z,Latasha Lopez
2,0967969634,2,3.98,West Susan,2025-09-14T00:44:57.000Z,Alan Wiley
3,4246141847,3,3.66,Mcintoshton,2025-08-26T22:28:17.000Z,James Taylor
4,617017010191777,4,3.86,North Courtneychester,2025-09-01T11:40:55.000Z,Theresa Benson
5,6110605683,5,4.87,Brownburgh,2025-09-04T16:35:04.000Z,Karen Jensen
6,5564809096439,6,4.26,Port Williamland,2025-08-31T14:31:37.000Z,Debra Smith
7,179856869529778,7,3.67,West Erinborough,2025-09-17T05:57:45.000Z,Justin Peters
8,706321839008097,8,4.9,Lake Stephen,2025-08-26T06:45:51.000Z,Todd Young
9,17279105045499,9,4.5,West Lindsey,2025-08-27T13:04:32.000Z,Mary Young
10,5096134480,10,4.04,Lauraland,2025-09-09T05:50:23.000Z,Jacob Mack


In [0]:
driver_obj = transformations(spark)
driver_df_transform = driver_obj.dedup(df_drivers,['driver_id'],'last_updated_timestamp')

display(driver_df_transform)

driver_id,phone_number,vehicle_id,driver_rating,city,last_updated_timestamp,full_name
1,2629242955590,1,4.7,East Dorothy,2025-08-25T06:36:26.000Z,Latasha Lopez
10,5096134480,10,4.04,Lauraland,2025-09-09T05:50:23.000Z,Jacob Mack
11,1430778973,11,3.58,Lake Stephenview,2025-08-20T18:43:00.000Z,Matthew Barajas
12,6948696563,12,4.91,New Michaelview,2025-08-25T18:14:33.000Z,Kimberly Sampson
13,3263292468,13,4.52,Strongfort,2025-09-09T09:45:17.000Z,David Silva
14,7192493764,14,4.83,South Tina,2025-09-18T20:52:07.000Z,Sean Christensen
15,2055683393,15,4.38,Petersenfurt,2025-09-01T07:01:10.000Z,Vanessa Williams
16,0012670315560,16,3.53,New Gabrielleside,2025-08-22T03:44:26.000Z,Ryan Henderson
17,6441682978787,17,3.83,Sydneystad,2025-08-30T15:09:25.000Z,Alfred Clayton
18,0159120292438,18,4.12,New Michaelshire,2025-09-13T08:53:24.000Z,Lisa Duarte


In [0]:
df_drivers=driver_obj.process_timestamp(driver_df_transform)
display(df_drivers)

driver_id,phone_number,vehicle_id,driver_rating,city,last_updated_timestamp,full_name,process_timestamp
1,2629242955590,1,4.7,East Dorothy,2025-08-25T06:36:26.000Z,Latasha Lopez,2026-06-06T19:00:29.641Z
10,5096134480,10,4.04,Lauraland,2025-09-09T05:50:23.000Z,Jacob Mack,2026-06-06T19:00:29.641Z
11,1430778973,11,3.58,Lake Stephenview,2025-08-20T18:43:00.000Z,Matthew Barajas,2026-06-06T19:00:29.641Z
12,6948696563,12,4.91,New Michaelview,2025-08-25T18:14:33.000Z,Kimberly Sampson,2026-06-06T19:00:29.641Z
13,3263292468,13,4.52,Strongfort,2025-09-09T09:45:17.000Z,David Silva,2026-06-06T19:00:29.641Z
14,7192493764,14,4.83,South Tina,2025-09-18T20:52:07.000Z,Sean Christensen,2026-06-06T19:00:29.641Z
15,2055683393,15,4.38,Petersenfurt,2025-09-01T07:01:10.000Z,Vanessa Williams,2026-06-06T19:00:29.641Z
16,0012670315560,16,3.53,New Gabrielleside,2025-08-22T03:44:26.000Z,Ryan Henderson,2026-06-06T19:00:29.641Z
17,6441682978787,17,3.83,Sydneystad,2025-08-30T15:09:25.000Z,Alfred Clayton,2026-06-06T19:00:29.641Z
18,0159120292438,18,4.12,New Michaelshire,2025-09-13T08:53:24.000Z,Lisa Duarte,2026-06-06T19:00:29.641Z


In [0]:
spark.sql("DROP TABLE IF EXISTS pysparkdbt.silver.drivers")

DataFrame[]

In [0]:
#spark.sql("DROP TABLE IF EXISTS pysparkdbt.silver.drivers")
spark.sql("DESCRIBE pysparkdbt.silver.customers").show()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|         customer_id|      int|   NULL|
|               email|   string|   NULL|
|        phone_number|   string|   NULL|
|                city|   string|   NULL|
|         signup_date|     date|   NULL|
|last_updated_time...|timestamp|   NULL|
|              domain|   string|   NULL|
|           full_name|   string|   NULL|
|   process_timestamp|timestamp|   NULL|
+--------------------+---------+-------+



In [0]:
from delta.tables import DeltaTable
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()  # ADD THIS

if not spark.catalog.tableExists('pysparkdbt.silver.drivers'):
    df_drivers.write.format('delta')\
        .mode('overwrite')\
        .saveAsTable("pysparkdbt.silver.drivers")
else:
    driver_obj.upsert(df_drivers, ["driver_id"], 'drivers', 'last_updated_timestamp')

##***Locations***

In [0]:
df_loc = spark.read.table('pysparkdbt.bronze.locations')


In [0]:
display(df_loc)

location_id,city,state,country,latitude,longitude,last_updated_timestamp
1,Lake Davidport,Hawaii,Dominica,34.5682,151.8097,2025-09-03T13:37:24.000Z
2,Mccarthybury,Minnesota,Dominican Republic,71.6928,-162.2836,2025-09-10T10:54:18.000Z
3,Bellhaven,Arkansas,Japan,-53.3075,69.1475,2025-09-17T06:52:51.000Z
4,Moorechester,New Hampshire,Togo,77.4281,57.0052,2025-08-27T19:04:27.000Z
5,Glennview,North Carolina,Gambia,41.1122,-162.4836,2025-08-20T17:31:06.000Z
6,Davilaville,Arizona,Antarctica (the territory South of 60 deg S),64.3413,-3.912,2025-09-05T13:22:58.000Z
7,Shawnfurt,New Jersey,Lithuania,71.1198,-143.6791,2025-09-03T20:35:10.000Z
8,Bradleytown,Florida,Monaco,30.0895,88.091,2025-08-29T19:45:05.000Z
9,East Miguel,Maryland,French Guiana,-52.9274,-56.9343,2025-08-27T23:43:41.000Z
10,Masonside,Wyoming,Saint Lucia,84.9905,81.5852,2025-09-06T20:25:27.000Z


In [0]:
loc_obj = transformations(spark)

In [0]:

loc_df_transform = loc_obj.dedup(df_loc,['location_id'],'last_updated_timestamp')
display(loc_df_transform)

location_id,city,state,country,latitude,longitude,last_updated_timestamp
1,Lake Davidport,Hawaii,Dominica,34.5682,151.8097,2025-09-03T13:37:24.000Z
10,Masonside,Wyoming,Saint Lucia,84.9905,81.5852,2025-09-06T20:25:27.000Z
11,Gregorytown,Connecticut,Venezuela,-20.4441,-110.9417,2025-09-19T15:34:25.000Z
12,Woodstown,North Dakota,Seychelles,-55.2071,54.4873,2025-09-16T02:31:19.000Z
13,North Susanstad,Wisconsin,Tunisia,29.7243,-32.3773,2025-08-22T18:01:13.000Z
14,North Jodimouth,Nevada,Togo,-56.7022,-94.9916,2025-08-26T08:31:22.000Z
15,Stevensstad,Missouri,Italy,-66.9045,133.5157,2025-08-26T16:59:13.000Z
16,New Josephburgh,Nebraska,Lesotho,-50.8801,1.9203,2025-09-15T17:26:33.000Z
17,Zacharyport,Utah,Peru,-63.2489,63.0064,2025-08-30T22:04:52.000Z
18,South Cynthia,Oklahoma,Cayman Islands,-8.8718,-27.5926,2025-09-18T16:00:47.000Z


In [0]:
df_loc = loc_obj.process_timestamp(loc_df_transform)

In [0]:
spark.sql("DROP TABLE IF EXISTS pysparkdbt.silver.drivers")

DataFrame[]

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()  # ADD THIS

if not spark.catalog.tableExists('pysparkdbt.silver.locations'):
    df_loc.write.format('delta')\
        .mode('append')\
        .saveAsTable("pysparkdbt.silver.locations")
else:
    loc_obj.upsert(df_loc, ["location_id"], 'locations', 'last_updated_timestamp')

In [0]:
spark.sql("DESCRIBE pysparkdbt.silver.customers").show()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|         customer_id|      int|   NULL|
|               email|   string|   NULL|
|        phone_number|   string|   NULL|
|                city|   string|   NULL|
|         signup_date|     date|   NULL|
|last_updated_time...|timestamp|   NULL|
|              domain|   string|   NULL|
|           full_name|   string|   NULL|
|   process_timestamp|timestamp|   NULL|
+--------------------+---------+-------+



##***Payments***

In [0]:
df_payments = spark.read.table('pysparkdbt.bronze.payments')


In [0]:
df_payments = df_payments.withColumn("onlinepayment_status",
    when((col('payment_method') == 'Card') & (col('payment_status') == 'Success'), 'Online Success')
    .when((col('payment_method') == 'Card') & (col('payment_status') == 'Failed'), 'Online Failed')
    .when((col('payment_method') == 'Card') & (col('payment_status') == 'Pending'), 'Online Pending')
    .otherwise('Offline')
)


In [0]:
display(df_payments)

payment_id,trip_id,customer_id,payment_method,payment_status,amount,transaction_time,last_updated_timestamp,onlinepayment_status
1,274,126,Cash,Success,38.15,2025-09-17T13:00:12.000Z,2025-08-30T13:40:53.000Z,Offline
2,676,131,Cash,Success,52.07,2025-08-14T13:00:12.000Z,2025-09-08T18:21:05.000Z,Offline
3,919,132,Card,Failed,55.5,2025-07-27T13:00:12.000Z,2025-08-21T20:24:08.000Z,Online Failed
4,247,34,Wallet,Pending,28.78,2025-07-27T13:00:12.000Z,2025-08-27T15:03:09.000Z,Offline
5,386,62,Card,Failed,55.02,2025-08-01T13:00:12.000Z,2025-09-08T23:15:06.000Z,Online Failed
6,834,78,Wallet,Success,27.43,2025-08-25T13:00:12.000Z,2025-09-20T09:03:05.000Z,Offline
7,348,144,Wallet,Failed,74.94,2025-08-26T13:00:12.000Z,2025-08-30T19:11:26.000Z,Offline
8,558,82,Card,Failed,19.58,2025-09-09T13:00:12.000Z,2025-09-06T05:53:26.000Z,Online Failed
9,260,151,Card,Pending,24.4,2025-07-23T13:00:12.000Z,2025-09-02T17:43:14.000Z,Online Pending
10,133,70,Cash,Failed,69.99,2025-07-31T13:00:12.000Z,2025-09-11T10:30:16.000Z,Offline


In [0]:
payments_obj = transformations(spark)
df_payments = payments_obj.process_timestamp(df_payments)
df_payments = payments_obj.dedup(df_payments,['payment_id'],'last_updated_timestamp')
display(df_payments)


payment_id,trip_id,customer_id,payment_method,payment_status,amount,transaction_time,last_updated_timestamp,onlinepayment_status,process_timestamp
1,274,126,Cash,Success,38.15,2025-09-17T13:00:12.000Z,2025-08-30T13:40:53.000Z,Offline,2026-06-06T19:01:57.625Z
10,133,70,Cash,Failed,69.99,2025-07-31T13:00:12.000Z,2025-09-11T10:30:16.000Z,Offline,2026-06-06T19:01:57.625Z
100,265,92,Wallet,Success,90.93,2025-07-15T13:00:12.000Z,2025-09-01T15:06:16.000Z,Offline,2026-06-06T19:01:57.625Z
1000,903,43,Card,Pending,96.61,2025-08-29T13:00:12.000Z,2025-08-30T01:59:42.000Z,Online Pending,2026-06-06T19:01:57.625Z
101,761,107,Cash,Success,8.75,2025-08-10T13:00:12.000Z,2025-09-01T13:16:51.000Z,Offline,2026-06-06T19:01:57.625Z
102,139,134,Wallet,Pending,59.42,2025-07-04T13:00:12.000Z,2025-08-25T19:45:46.000Z,Offline,2026-06-06T19:01:57.625Z
103,604,80,Card,Success,45.54,2025-09-17T13:00:12.000Z,2025-09-13T02:33:29.000Z,Online Success,2026-06-06T19:01:57.625Z
104,434,122,Cash,Failed,73.37,2025-07-04T13:00:12.000Z,2025-09-04T19:07:43.000Z,Offline,2026-06-06T19:01:57.625Z
105,168,80,Cash,Pending,68.93,2025-08-30T13:00:12.000Z,2025-08-30T18:12:56.000Z,Offline,2026-06-06T19:01:57.625Z
106,814,87,Card,Failed,29.59,2025-08-02T13:00:12.000Z,2025-09-19T08:05:12.000Z,Online Failed,2026-06-06T19:01:57.625Z


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()  # ADD THIS

if not spark.catalog.tableExists('pysparkdbt.silver.payments'):
    df_payments.write.format('delta')\
        .mode('append')\
        .saveAsTable("pysparkdbt.silver.payments")
else:
    payments_obj.upsert(df_payments, ["payment_id"], 'payments', 'last_updated_timestamp')

In [0]:
display(df_payments)

payment_id,trip_id,customer_id,payment_method,payment_status,amount,transaction_time,last_updated_timestamp,onlinepayment_status,process_timestamp
1,274,126,Cash,Success,38.15,2025-09-17T13:00:12.000Z,2025-08-30T13:40:53.000Z,Offline,2026-06-06T19:02:05.335Z
10,133,70,Cash,Failed,69.99,2025-07-31T13:00:12.000Z,2025-09-11T10:30:16.000Z,Offline,2026-06-06T19:02:05.335Z
100,265,92,Wallet,Success,90.93,2025-07-15T13:00:12.000Z,2025-09-01T15:06:16.000Z,Offline,2026-06-06T19:02:05.335Z
1000,903,43,Card,Pending,96.61,2025-08-29T13:00:12.000Z,2025-08-30T01:59:42.000Z,Online Pending,2026-06-06T19:02:05.335Z
101,761,107,Cash,Success,8.75,2025-08-10T13:00:12.000Z,2025-09-01T13:16:51.000Z,Offline,2026-06-06T19:02:05.335Z
102,139,134,Wallet,Pending,59.42,2025-07-04T13:00:12.000Z,2025-08-25T19:45:46.000Z,Offline,2026-06-06T19:02:05.335Z
103,604,80,Card,Success,45.54,2025-09-17T13:00:12.000Z,2025-09-13T02:33:29.000Z,Online Success,2026-06-06T19:02:05.335Z
104,434,122,Cash,Failed,73.37,2025-07-04T13:00:12.000Z,2025-09-04T19:07:43.000Z,Offline,2026-06-06T19:02:05.335Z
105,168,80,Cash,Pending,68.93,2025-08-30T13:00:12.000Z,2025-08-30T18:12:56.000Z,Offline,2026-06-06T19:02:05.335Z
106,814,87,Card,Failed,29.59,2025-08-02T13:00:12.000Z,2025-09-19T08:05:12.000Z,Online Failed,2026-06-06T19:02:05.335Z


In [0]:
%sql
select count(*) from pysparkdbt.silver.payments

count(*)
1000


###***vehicles***


In [0]:
df_vehicles = spark.read.table('pysparkdbt.bronze.vehicles')
display(df_vehicles)


vehicle_id,license_plate,model,make,year,vehicle_type,last_updated_timestamp
1,NXT-8646,Message,"Francis, Smith and Lee",2023,Hatchback,2025-09-02T06:05:20.000Z
2,03S R43,Region,Lawson Group,2017,Sedan,2025-08-31T05:55:09.000Z
3,SKO H06,Prepare,"Moreno, Ruiz and Barker",2023,Luxury,2025-08-30T05:07:29.000Z
4,5R235,Pattern,"Welch, Martinez and Hendricks",2019,Van,2025-09-09T05:52:10.000Z
5,925A,Process,Rivera-Anderson,2014,Hatchback,2025-08-27T11:42:16.000Z
6,6-1130V,On,Brown Ltd,2014,SUV,2025-09-06T06:37:52.000Z
7,L25-TWN,Plan,"Gonzalez, Rios and Rios",2020,Sedan,2025-09-15T13:14:48.000Z
8,GWY 958,Month,"Smith, Mckenzie and Bullock",2017,Sedan,2025-09-11T16:20:12.000Z
9,4FG 919,Speak,"Rice, Barnes and Hernandez",2019,SUV,2025-09-11T02:43:13.000Z
10,6FD 648,Religious,Schwartz and Sons,2017,SUV,2025-09-11T19:10:01.000Z


In [0]:
df_vehicles = df_vehicles.withColumn("make",upper(col("make")))

In [0]:
display(df_vehicles)

vehicle_id,license_plate,model,make,year,vehicle_type,last_updated_timestamp
1,NXT-8646,Message,"FRANCIS, SMITH AND LEE",2023,Hatchback,2025-09-02T06:05:20.000Z
2,03S R43,Region,LAWSON GROUP,2017,Sedan,2025-08-31T05:55:09.000Z
3,SKO H06,Prepare,"MORENO, RUIZ AND BARKER",2023,Luxury,2025-08-30T05:07:29.000Z
4,5R235,Pattern,"WELCH, MARTINEZ AND HENDRICKS",2019,Van,2025-09-09T05:52:10.000Z
5,925A,Process,RIVERA-ANDERSON,2014,Hatchback,2025-08-27T11:42:16.000Z
6,6-1130V,On,BROWN LTD,2014,SUV,2025-09-06T06:37:52.000Z
7,L25-TWN,Plan,"GONZALEZ, RIOS AND RIOS",2020,Sedan,2025-09-15T13:14:48.000Z
8,GWY 958,Month,"SMITH, MCKENZIE AND BULLOCK",2017,Sedan,2025-09-11T16:20:12.000Z
9,4FG 919,Speak,"RICE, BARNES AND HERNANDEZ",2019,SUV,2025-09-11T02:43:13.000Z
10,6FD 648,Religious,SCHWARTZ AND SONS,2017,SUV,2025-09-11T19:10:01.000Z


In [0]:
vehicle_oj = transformations(spark)

df_vehicles = vehicle_oj.dedup(df_vehicles,['vehicle_id'],'last_updated_timestamp')
df_vehicles = vehicle_oj.process_timestamp(df_vehicles)
display(df_vehicles)
from delta.tables import DeltaTable
from pyspark.sql import SparkSession

if not spark.catalog.tableExists('pysparkdbt.silver.vehicles'):
    df_vehicles.write.format('delta')\
        .mode('append')\
        .saveAsTable("pysparkdbt.silver.vehicles")
else:
    vehicle_oj.upsert(df_vehicles, ["vehicle_id"], 'vehicles', 'pysparkdbt.silver')



vehicle_id,license_plate,model,make,year,vehicle_type,last_updated_timestamp,process_timestamp
1,NXT-8646,Message,"FRANCIS, SMITH AND LEE",2023,Hatchback,2025-09-02T06:05:20.000Z,2026-06-06T19:02:12.028Z
10,6FD 648,Religious,SCHWARTZ AND SONS,2017,SUV,2025-09-11T19:10:01.000Z,2026-06-06T19:02:12.028Z
11,0N 8334L,Name,EVANS-KING,2010,SUV,2025-09-10T19:40:04.000Z,2026-06-06T19:02:12.028Z
12,3YH 743,So,MARTINEZ AND SONS,2011,Sedan,2025-09-03T19:53:15.000Z,2026-06-06T19:02:12.028Z
13,ULA0960,Event,REED INC,2010,Sedan,2025-08-31T05:38:52.000Z,2026-06-06T19:02:12.028Z
14,982MCB,Road,CHAVEZ GROUP,2014,Luxury,2025-08-25T11:03:53.000Z,2026-06-06T19:02:12.028Z
15,0I 6R0MOA,Item,ROBBINS LLC,2015,Sedan,2025-08-20T16:17:49.000Z,2026-06-06T19:02:12.028Z
16,6WJ63,Store,JOHNSON-PRINCE,2017,Van,2025-09-07T09:41:02.000Z,2026-06-06T19:02:12.028Z
17,542-VGUN,East,"PEREZ, LEE AND HARRIS",2023,Sedan,2025-09-02T18:06:22.000Z,2026-06-06T19:02:12.028Z
18,RLY 8125,Low,"LUNA, BRYANT AND RHODES",2019,Luxury,2025-09-19T17:46:03.000Z,2026-06-06T19:02:12.028Z


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5670871397449901>, line 14
     10     df_vehicles.write.format('delta')\
     11         .mode('append')\
     12         .saveAsTable("pysparkdbt.silver.vehicles")
     13 else:
---> 14     vehicle_oj.upsert(df_vehicles, ["vehicle_id"], 'vehicles', 'pysparkdbt.silver')

File /Workspace/Users/sahudeepika471@gmail.com/pyspark_dbt_proj/utils/custom_utils.py:28, in transformations.upsert(self, df, key_cols, table, cdc)
     26 merge_condition = " AND ".join([f"t.{k} = s.{k}" for k in key_cols])
     27 dlt_obj = DeltaTable.forName(self.spark,f"pysparkdbt.silver.{table}")
---> 28 dlt_obj.alias("t").merge(df.alias("s"), merge_condition).whenMatchedUpdateAll(condition=f"s.{cdc}>=t.{cdc}").whenNotMatchedInsertAll().execute()
     30 return 1

File /databricks/python/lib/python3.12/site-packages/delta/connect/tables.py:609, in De

In [0]:
%sql
select count(*) from pysparkdbt.silver.vehicles

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:726)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:444)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:444)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can